# ASG Airlines — Raw to Bronze Ingestion

**Stage:** Raw → Bronze (Medallion Architecture)
**Storage:** Azure Data Lake Storage Gen2 (`az://` protocol), account `stasgairlines01`
**Execution mode:** Native Python + Delta Lake (`deltalake` / delta-rs), not PySpark — see the project documentation, Section 14, for why Databricks/PySpark compute was substituted with local execution.

## What this notebook does

1. Downloads the single source Excel workbook (`UseCase_-_Airlines.xlsx`) from the ADLS `raw` container.
2. Parses all 4 sheets (`flights`, `bookings`, `passengers`, `payments`) into DataFrames, reading every column as **string type** — this is a deliberate, non-obvious choice, explained below.
3. Runs an ingestion-time data quality assessment (nulls, duplicates, dtypes) **without changing any data** — Bronze must stay a faithful copy of the source.
4. Writes each sheet as a Delta Lake table into the `bronze` container.
5. Reads the tables back to verify the write succeeded.

### Why every column is read as `dtype=str`

An earlier version of this script let pandas infer column types automatically, which silently converted `passengers.aadhaar_id` to `int64` — and int64 strips leading zeros. That corrupted 198 of 1,039 Aadhaar numbers (a government ID) before we'd even reached the cleaning stage. Reading everything as string at ingestion avoids this entire class of bug, and is safer for Bronze generally: Bronze's job is to preserve the source exactly, and inferred numeric types can silently lose information (leading zeros, exact string formatting) that later stages may need.

## Step 1 — Storage account & environment configuration

The storage key is read from the `ADLS_STORAGE_KEY` environment variable; if it isn't set, we fall back to fetching it live via the Azure CLI. The key itself is never hardcoded or logged.

In [1]:
import os
import io
import datetime
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient
from deltalake import write_deltalake, DeltaTable

STORAGE_ACCOUNT_NAME = "stasgairlines01"
CONTAINER_RAW        = "raw"
CONTAINER_BRONZE     = "bronze"
RAW_FILE_NAME        = "UseCase_-_Airlines.xlsx"

STORAGE_KEY = os.environ.get("ADLS_STORAGE_KEY", "")
if not STORAGE_KEY:
    try:
        import subprocess
        res = subprocess.run(
            ["az", "storage", "account", "keys", "list",
             "--account-name", STORAGE_ACCOUNT_NAME,
             "--resource-group", "rg-asg-airlines",
             "--query", "[0].value", "-o", "tsv"],
            capture_output=True, text=True, check=True
        )
        STORAGE_KEY = res.stdout.strip()
    except Exception:
        pass

if not STORAGE_KEY:
    raise RuntimeError(
        "ADLS_STORAGE_KEY environment variable is not set and az CLI key lookup failed. "
        "Run: $env:ADLS_STORAGE_KEY = (az storage account keys list ...)"
    )

print(f"Storage account: {STORAGE_ACCOUNT_NAME}")

Storage account: stasgairlines01


## Step 2 — Download the source Excel workbook from the `raw` container

In [1]:
service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    credential=STORAGE_KEY
)
raw_file_client = service_client.get_file_system_client(CONTAINER_RAW).get_file_client(RAW_FILE_NAME)
download_stream = raw_file_client.download_file()
excel_bytes = download_stream.readall()

print(f"Downloaded {len(excel_bytes):,} bytes")

excel_file = pd.ExcelFile(io.BytesIO(excel_bytes))
sheet_names = ["flights", "bookings", "passengers", "payments"]
print(f"Workbook sheets found: {excel_file.sheet_names}")

Downloaded 277,887 bytes
Workbook sheets found: ['flights', 'bookings', 'passengers', 'payments']


## Step 3 — Parse sheets and attach ingestion metadata

Every column is read as `dtype=str` (see rationale above). Null-like string artifacts (`'nan'`, `'None'`, `'<NA>'`, `'NaT'`, empty string) are normalized to a true `None` so downstream null-counting is accurate. Two audit columns are added to every table: `ingestion_timestamp` and `source_file`.

In [1]:
ingestion_time = datetime.datetime.now(datetime.timezone.utc)
raw_dfs = {}

for sheet in sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet, dtype=str)

    for col in df.columns:
        df[col] = df[col].apply(lambda x: str(x).strip() if pd.notna(x) else None)
        df[col] = df[col].replace({"nan": None, "None": None, "<NA>": None, "NaT": None, "": None})

    df["ingestion_timestamp"] = ingestion_time
    df["source_file"] = RAW_FILE_NAME

    raw_dfs[sheet] = df
    print(f"Loaded '{sheet}': {len(df):,} rows, {len(df.columns)-2} raw columns + 2 metadata columns")

Loaded 'flights': 1,020 rows, 7 raw columns + 2 metadata columns
Loaded 'bookings': 1,000 rows, 9 raw columns + 2 metadata columns
Loaded 'passengers': 1,039 rows, 9 raw columns + 2 metadata columns
Loaded 'payments': 1,000 rows, 4 raw columns + 2 metadata columns


## Step 4 — Ingestion-time data quality assessment

This is a **read-only profiling pass** — nothing here changes the data. The goal is to find real, specific issues (not assumed ones) before any cleaning decisions are made. For each table we report per-column null counts/percentages and a duplicate-row check across the original source columns (excluding the audit columns we just added).

In [1]:
print("=" * 80)
print("ASG AIRLINES — INGESTION-TIME DATA QUALITY REPORT")
print("=" * 80)

for sheet, df in raw_dfs.items():
    total_rows = len(df)
    source_cols = [c for c in df.columns if c not in ["ingestion_timestamp", "source_file"]]

    print(f"\nTABLE: {sheet.upper()}  (Total Ingested Rows: {total_rows:,})")
    print(f"{'Column Name':<35} | {'Null Count':>10} | {'Null %':>8} | Data Type")
    print("-" * 72)
    for col in source_cols:
        null_cnt = df[col].isna().sum()
        null_pct = (null_cnt / total_rows * 100) if total_rows > 0 else 0
        print(f"{col:<35} | {null_cnt:>10,} | {null_pct:>7.2f}% | {str(df[col].dtype)}")

    distinct_cnt = len(df[source_cols].drop_duplicates())
    dup_cnt = total_rows - distinct_cnt
    dup_pct = (dup_cnt / total_rows * 100) if total_rows > 0 else 0
    print(f"\nDuplicate Row Check: Distinct={distinct_cnt:,} | Duplicates={dup_cnt:,} ({dup_pct:.2f}%)")

ASG AIRLINES — INGESTION-TIME DATA QUALITY REPORT

TABLE: FLIGHTS  (Total Ingested Rows: 1,020)
Column Name                         | Null Count |   Null % | Data Type
------------------------------------------------------------------------
flight_id                           |          0 |    0.00% | object
airline                             |         41 |    4.02% | object
source                              |          0 |    0.00% | object
destination                         |          0 |    0.00% | object
departure_time                      |          0 |    0.00% | object
arrival_time                        |          0 |    0.00% | object
duration                            |          0 |    0.00% | object

Duplicate Row Check: Distinct=1,005 | Duplicates=15 (1.47%)

TABLE: BOOKINGS  (Total Ingested Rows: 1,000)
Column Name                         | Null Count |   Null % | Data Type
------------------------------------------------------------------------
booking_id             

## Step 5 — Write Delta Lake tables to the `bronze` container

Each sheet becomes its own Delta table. `mode="overwrite"` with `schema_mode="overwrite"` means re-running this notebook is idempotent — it fully replaces Bronze rather than appending duplicate ingestion runs.

In [1]:
storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT_NAME,
    "azure_storage_access_key": STORAGE_KEY,
}

for sheet, df in raw_dfs.items():
    delta_uri = f"az://{CONTAINER_BRONZE}/{sheet}"
    write_deltalake(delta_uri, df, mode="overwrite", schema_mode="overwrite", storage_options=storage_options)
    print(f"Wrote {len(df):,} rows -> {delta_uri}")

Wrote 1,020 rows -> az://bronze/flights
Wrote 1,000 rows -> az://bronze/bookings
Wrote 1,039 rows -> az://bronze/passengers
Wrote 1,000 rows -> az://bronze/payments


## Step 6 — Verify the Bronze tables by reading them back

In [1]:
for sheet in sheet_names:
    delta_uri = f"az://{CONTAINER_BRONZE}/{sheet}"
    dt = DeltaTable(delta_uri, storage_options=storage_options)
    verified_df = dt.to_pandas()
    print(f"[Verified] bronze/{sheet}: {len(verified_df):,} rows | Schema fields: {len(dt.schema().fields)}")

print("\nIngestion to Bronze layer complete.")

[Verified] bronze/flights: 1,020 rows | Schema fields: 9
[Verified] bronze/bookings: 1,000 rows | Schema fields: 11
[Verified] bronze/passengers: 1,039 rows | Schema fields: 11
[Verified] bronze/payments: 1,000 rows | Schema fields: 6

Ingestion to Bronze layer complete.
